# Model Adequacy, Lack of Fit and Practice

**STA 506 2.0 Linear Regression Analysis** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta506.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [What validation asks](#theory-01)
- [Residual diagnostics](#theory-02)
- [Outliers, leverage and influence](#theory-03)
- [Replication and pure error](#theory-04)
- [Lack-of-fit decomposition](#theory-05)
- [Remedies and validation data](#theory-06)
- [Prediction and interpretation](#theory-07)

## Detailed theory · Model adequacy and lack of fit

<a id="theory-01"></a>

### What validation asks
A fitted model can compute coefficients even when its assumptions are unsuitable. Assess the conditional mean form, error dependence, variance, distribution for inference, influential cases and prediction performance. Numerical summaries alone do not validate a model.

<a id="theory-02"></a>

### Residual diagnostics
Plot residuals against fitted values and predictors for curvature and funnels; against run or time order for dependence; and on a Q–Q plot for distributional departures. Residuals are fitted quantities with constraints, not an independent observed error sample. A roughly straight Q–Q pattern is compatible with normality, not proof of it.

<a id="theory-03"></a>

### Outliers, leverage and influence
A response outlier has an unusual response conditional on predictors. Leverage measures unusual predictor position; in an OLS design the diagonal of $H=X(X^\top X)^{-1}X^\top$ gives h_ii. Influence describes how much a fitted result changes when an observation is removed or perturbed. A high-leverage point may fit closely yet strongly control a slope. Investigate data provenance before deletion.

<a id="theory-04"></a>

### Replication and pure error
With g distinct predictor settings and $n_j$ replicates at setting j, define the setting mean $\bar y_j$:
$$SS_{PE}=\sum_j\sum_i(y_{ij}-\bar y_j)^2,\qquad df_{PE}=n-g.$$
Pure error estimates variation among replicates sharing the same predictors. Without replication it cannot be separately estimated by this classical procedure.

<a id="theory-05"></a>

### Lack-of-fit decomposition
For a regression with p fitted coefficients,
$$SS_{LOF}=\sum_j n_j(\bar y_j-\hat y_j)^2=SSE-SS_{PE},\quad df_{LOF}=g-p.$$
Under an adequate mean function with common independent normal errors,
$$F_{LOF}=\frac{SS_{LOF}/(g-p)}{SS_{PE}/(n-g)}.$$
Require positive numerator and denominator df. Use the upper F tail. Large F indicates systematic departure from the fitted form; a nonsignificant result may simply reflect limited power.

<a id="theory-06"></a>

### Remedies and validation data
Possible responses include a scientifically justified transformation, nonlinear terms, an omitted factor, variance modelling or a method appropriate for correlated errors. Choose according to the diagnosed issue. Validation should respect time, clustering and groups; random row splitting can leak information when nearby records depend on one another.

<a id="theory-07"></a>

### Prediction and interpretation
An apparently adequate model can fail outside the predictor support or in a new population. Report the settings with replication, pure-error and lack-of-fit df, diagnostics, choices made after inspecting results and limits of prediction. Do not use data-dependent deletion or model searching as though the final p-value came from a predeclared model.

## Concepts covered

**Course points:** Residual diagnostics; assumptions and influence; pure error and lack of fit; model repair.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Lack of fit: separate pure error from model failure

### Question and assumptions

Fit a straight line to repeated predictor settings x=[1,1,2,2,3,3], y=[1,3,2,4,7,9].

### Procedure

1. Fit the linear regression.
2. At each repeated x, compute the within-setting mean and pure-error squared deviations.
3. Subtract pure error from regression SSE, then construct the lack-of-fit ANOVA table.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| x | y | Mean at x | Linear fitted y | Pure-error contribution | Regression-error contribution |
| --- | --- | --- | --- | --- | --- |
| 1 | 1 | 2 | 1.33333 | 1 | 0.111111 |
| 1 | 3 | 2 | 1.33333 | 1 | 2.77778 |
| 2 | 2 | 3 | 4.33333 | 1 | 5.44444 |
| 2 | 4 | 3 | 4.33333 | 1 | 0.111111 |
| 3 | 7 | 8 | 7.33333 | 1 | 0.111111 |
| 3 | 9 | 8 | 7.33333 | 1 | 2.77778 |

| Source | SS | df | MS | F |
| --- | --- | --- | --- | --- |
| Lack of fit | 5.33333 | 1 | 5.33333 | 2.66667 |
| Pure error | 6 | 3 | 2 |  |
| Regression residual | 11.3333 | 4 | 2.83333 |  |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


x=[1,1,2,2,3,3]; y=[1,3,2,4,7,9]; xb=mean(x); yb=mean(y)
b1=sum((a-xb)*(b-yb) for a,b in zip(x,y))/sum((a-xb)**2 for a in x); b0=yb-b1*xb
rows=[]
for a,b in zip(x,y):
    groupmean=mean(v for z,v in zip(x,y) if z==a); fitted=b0+b1*a
    rows.append([a,b,groupmean,fitted,(b-groupmean)**2,(b-fitted)**2])
show_table(["x","y","Mean at x","Linear fitted y","Pure-error contribution","Regression-error contribution"],rows)
PE=sum(r[4] for r in rows); SSE=sum(r[5] for r in rows); LOF=SSE-PE
n=len(y); g=len(set(x)); p=2; dfPE=n-g; dfLOF=g-p
F=(LOF/dfLOF)/(PE/dfPE)
show_table(["Source","SS","df","MS","F"],[["Lack of fit",LOF,dfLOF,LOF/dfLOF,F],["Pure error",PE,dfPE,PE/dfPE,""],["Regression residual",SSE,n-p,SSE/(n-p),""]])
assert abs(SSE-LOF-PE)<1e-12 and dfLOF+dfPE==n-p


### Decision and interpretation

The lack-of-fit test needs replicated predictor settings to estimate pure error. Compare F with an F(1,3) upper-tail reference for this example. A large value suggests the straight-line mean structure is inadequate; a small value does not prove the model is correct.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
x=np.repeat(np.arange(1,9),5); y=4+1.1*x+.65*x*x+rng.normal(0,4,len(x)); X=np.c_[np.ones(len(x)),x]
b=np.linalg.lstsq(X,y,rcond=None)[0]; fit=X@b; SSE=np.sum((y-fit)**2); levels=np.unique(x)
PE=sum(np.sum((y[x==g]-y[x==g].mean())**2) for g in levels); LOF=SSE-PE
F=(LOF/(len(levels)-2))/(PE/(len(y)-len(levels)))
print('SSE',round(SSE,1),'pure error',round(PE,1),'lack of fit',round(LOF,1),'F',round(F,2))
plt.scatter(fit,y-fit); plt.axhline(0,color='k',ls='--'); plt.show()

### Practice set · Model crash lab
1. At x=2, three replicate responses are 8, 10, 12. Calculate that setting’s pure-error SS.
2. If SSE=100 and SS_PE=40, what is SS_LOF?
3. Name two patterns a residual-versus-fitted plot can reveal.
4. Distinguish leverage from influence.

<details><summary>Answer sketches</summary>
1. Mean 10; SS_PE=(8−10)²+(10−10)²+(12−10)²=8. 2. 60. 3. Curvature, fan-shaped spread, clusters, or omitted structure. 4. Leverage is unusual X location; influence is impact on fitted estimates, often combining leverage and residual size.
</details>

### Debrief
Why does high R-squared not prove adequacy or causation?

Record the formula, assumptions, and one follow-up question.

## Lab reflection

Which assumptions came from the study design? What changed when you adjusted a parameter?